# LAV-DF private, authenticated baseline workflow

**Unexecuted template: no dataset or training is bundled.** First personally review and accept [LAV-DF terms](https://github.com/ControlNet/LAV-DF/blob/master/TERMS_AND_CONDITIONS.md) and the referenced VoxCeleb2 agreements. Confirm that your provider/institution permits Colab, private storage, model training and a faculty demonstration. The code cannot accept those terms or verify legal rights. Do not commit or publish tokens, videos, features, provenance, private metadata, reports or weights.

Use the setup cell below to check out the exact reviewed commit and install the pinned CPU environment. Set an existing private root outside the checkout. The original dataset host and exact file layout have not been tested here.


In [ ]:
import os, re, subprocess, sys
from pathlib import Path

REPO_COMMIT = '9d640ea5dec0c66e8e7051f6ae3ac0307bfdf57a'  # exact 40-character SHA of the reviewed branch commit
REPO = Path('/content/deepfake-detection-dtw')
VENV = Path('/content/deepfake-venv')
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT), 'Set the reviewed repository commit SHA'
assert sys.version_info[:2] == (3, 12), 'This pinned environment requires Python 3.12'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Pradeepbeleri/deepfake-detection-dtw.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'switch', '--detach', REPO_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == REPO_COMMIT
subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '--index-url', 'https://download.pytorch.org/whl/cpu', 'torch==2.5.1+cpu', 'torchvision==0.20.1+cpu'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-c', str(REPO / 'requirements-lock.txt'), '-e', str(REPO)], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)


In [ ]:
import os, json, subprocess, sys
from pathlib import Path

REPO = Path('/content/deepfake-detection-dtw')
PYTHON = '/content/deepfake-venv/bin/python'
PRIVATE = Path('/content/drive/MyDrive/deepfake-private')  # only if provider/institution authorize this storage
MEDIA = PRIVATE / 'lavdf' / 'media'
FEATURES = PRIVATE / 'lavdf' / 'features'
REPORTS = PRIVATE / 'lavdf' / 'reports'
PROVENANCE = PRIVATE / 'lavdf' / 'curated-provenance.json'
PLAN = PRIVATE / 'lavdf' / 'plan.json'
METADATA_LOCAL = PRIVATE / 'lavdf' / 'metadata.min.json'
WHISPER = PRIVATE / 'tiny.en.pt'
DINO = PRIVATE / 'dinov2_vits14_pretrain.pth'
DINO_SOURCE = PRIVATE / 'dinov2-src'  # clean pinned official checkout
DINO_SHA256 = ''  # full hash of your officially obtained file, verified after transfer
assert (REPO / 'src/deepfake_dtw/lavdf.py').exists() and Path(PYTHON).is_file(), 'Complete repository environment setup first'
assert PRIVATE.is_dir(), 'Mount and create your authorized private storage root before continuing'
assert len(DINO_SHA256) == 64 and WHISPER.is_file() and DINO.is_file(), 'Provide and verify both permitted model files'
for directory in (MEDIA, FEATURES, REPORTS):
    directory.mkdir(parents=True, mode=0o700, exist_ok=True)


## Provider access (interactive)

In a Colab cell, install `huggingface_hub==0.30.2` if needed and call `huggingface_hub.login(add_to_git_credential=False)`. Enter a read-only token only into the provider's secure prompt, never in a notebook cell or chat. Set `HF_HOME` to an owner-only ephemeral directory before login; clear it after use. Review the provider dataset card, confirm access and whether direct per-file downloads are available. Do not download the full dataset by default. If the repository exposes only archives, follow the provider's approved selective extraction instructions and audit archive paths; do not assume the filenames below exist.


In [ ]:
# Run only after agreeing to provider terms and confirming your allowed storage.
os.environ['HF_HOME'] = '/content/lavdf-hf-private'
Path(os.environ['HF_HOME']).mkdir(mode=0o700, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub==0.30.2'], check=True)
from huggingface_hub import login, HfApi, hf_hub_download
login(add_to_git_credential=False)
files = HfApi().list_repo_files('ControlNet/LAV-DF', repo_type='dataset')
print('Provider file count:', len(files))
print('Candidate metadata filenames:', [x for x in files if 'metadata' in x.lower()][:20])


## Obtain metadata and curate provenance

Choose the exact official metadata filename after inspecting the listing. If no JSON metadata is directly downloadable, obtain it using the official provider instructions. The sidecar is a private JSON dictionary keyed by exact provider `file` path. Each entry must contain `source_identity`, `target_identity` (required for manipulated examples), `source_recordings` (include provider `original` exactly for manipulated examples), and `usage_rights`. Identity and source links must come from audited provider evidence; do not infer or fabricate them. The official metadata itself does not establish all these fields.


In [ ]:
METADATA_FILENAME = ''  # exact provider filename, selected after listing
assert METADATA_FILENAME in files and METADATA_FILENAME.endswith('.json')
metadata_path = hf_hub_download(repo_id='ControlNet/LAV-DF', repo_type='dataset', filename=METADATA_FILENAME, local_dir=str(MEDIA))
assert PROVENANCE.is_file(), 'Supply the independently audited private provenance sidecar before planning'
assert not PLAN.exists(), 'Choose a new plan path; no overwrites'
subprocess.run([PYTHON, '-m', 'deepfake_dtw.lavdf', '--metadata', metadata_path, '--provenance', str(PROVENANCE), '--media-root', str(MEDIA), '--feature-root', str(FEATURES), '--output', str(PLAN), '--limit-per-split', '60', '--seed', '42'], cwd=REPO, check=True)


## Download only the selected official media

The plan preserves official splits and whole identity/source groups. If listed filenames are not direct Hugging Face repository files, stop and use an authorized provider-supported archive workflow; do not download arbitrary or unlisted mirrors. Check available storage and identity overlaps before proceeding.


In [ ]:
records = json.loads(PLAN.read_text())
selected = sorted({r['provider_file'] for r in records})
missing = [name for name in selected if name not in files]
assert not missing, 'Selected videos are not direct provider files; use official authorized archive access instead'
for name in selected:
    hf_hub_download(repo_id='ControlNet/LAV-DF', repo_type='dataset', filename=name, local_dir=str(MEDIA))
print('Downloaded selected files:', len(selected))


## Extract and audit before training

This CPU path may be slow and memory intensive. It must use the verified pretrained weights, and each failure halts the run; never relabel or silently discard failed records. Resolve extraction failures and re-audit provenance and split integrity before training. The JSON reports and features contain sensitive metadata; keep them private.


In [ ]:
env = dict(os.environ, OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', XFORMERS_DISABLED='1')
for r in records:
    out = Path(r['feature_path']); report = REPORTS / (r['sample_id'] + '.json')
    assert not out.exists() and not report.exists(), 'Choose fresh private outputs'
    with report.open('x') as stream:
        subprocess.run([PYTHON, '-m', 'deepfake_dtw.extract', r['video_path'], '--audio-checkpoint', str(WHISPER), '--visual-checkpoint', str(DINO), '--visual-format', 'dinov2_vits14', '--dinov2-source', str(DINO_SOURCE), '--visual-sha256', DINO_SHA256, '--authorized-models', '--device', 'cpu', '--output', str(out)], cwd=REPO, env=env, stdout=stream, check=True)
print('Feature extraction completed for all planned clips; inspect private quality reports.')


## Train and final evaluation

Audit labels, real/fake and manipulation counts, identities, original links, audio and provider split integrity before running. The trainer checks metadata and file hashes for all partitions and reads only training and validation feature arrays during fitting. Validation selects both epoch and threshold. Run final held-out evaluation only after all decisions are frozen. No score is calibrated by this workflow.


In [ ]:
checkpoint = PRIVATE / 'lavdf' / 'baseline.pt'
training_report = REPORTS / 'training.json'
assert not checkpoint.exists() and not training_report.exists()
with training_report.open('x') as stream:
    subprocess.run([PYTHON, '-m', 'deepfake_dtw.training', 'train', str(PLAN), str(checkpoint), '--seed', '42', '--epochs', '10'], cwd=REPO, env=env, stdout=stream, check=True)
print('Training report saved privately; review validation before final evaluation.')


In [ ]:
# Run only after freezing all decisions. This writes sensitive failure IDs privately.
report = REPORTS / 'held-out-test.json'
assert checkpoint.is_file() and not report.exists()
with report.open('x') as stream:
    subprocess.run([PYTHON, '-m', 'deepfake_dtw.training', 'evaluate', str(PLAN), str(checkpoint), '--partition', 'test'], cwd=REPO, env=env, stdout=stream, check=True)
print('Held-out report saved privately.')
